Simple GenAI Application

In [ ]:
# Load the environment variables

from dotenv import load_dotenv
import os

load_dotenv(override=True)


# This setup is required to log events into Langsmith
os.environ["API_HOST"] = os.getenv("API_HOST", "github")
# Langchain Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "")
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2")

API_HOST = os.getenv("API_HOST", "github")

## Load Data --> Docs --> Divide into Chunks --> Text --> Convert into Vectors Embeddings --> vectors --> store in Vector Stores

In [ ]:
# Load Data 
from langchain_community.document_loaders import WebBaseLoader
import bs4

web_loader = WebBaseLoader("https://docs.langchain.com/oss/python/langchain/overview")
docs = web_loader.load()

In [ ]:
## Divide into Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

final_docs = text_splitter.split_documents(docs)
print(f"No of chunks: {len(final_docs)}")

In [ ]:
## Converting into Vectors - using OpenAI Embeddings

from langchain_openai import OpenAIEmbeddings
#embeddings = OpenAIEmbeddings(model='text-embedding-3-large') """This code will work if you have an OpenAI account and API Key"""

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["GITHUB_TOKEN"],  # Your GitHub PAT
    base_url="https://models.inference.ai.azure.com")

In [ ]:
## Store the vectors in FAISS database or Chroma
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(final_docs, embedding=embedding_model)

In [ ]:
query = "what are the core benefits of langsmth?"
response = vectorstoredb.similarity_search(query)
response

In [ ]:
## Retrieval Chain, Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Define the LLM model
if API_HOST == "github":
    print("Using GitHub models...and model:", os.getenv("GITHUB_MODEL", "openai/gpt-4o"))
    llm = ChatOpenAI(
        model_name=os.getenv("GITHUB_MODEL", "openai/gpt-4o"),
        openai_api_base="https://models.github.ai/inference",
        openai_api_key=os.environ["GITHUB_TOKEN"],
    )
elif API_HOST == "ollama":
    print("Using Ollama model on local...")
    llm = ChatOpenAI(
        model_name=os.getenv("OLLAMA_MODEL", "mistral"),
        openai_api_base=os.environ["OLLAMA_ENDPOINT"],
        openai_api_key="nokeyneeded",
    )



In [ ]:
# Define the prompt with context

query = "what are the core benefits of langsmth?"

prompt = ChatPromptTemplate.from_template(
"""
Answer the following based on the context.
<context>
{context}
</context>
"""
)

document_chain = create_stuff_documents_chain(llm, prompt) ## This combines and passes the docs inform of context

# Method 1: Directly send the context by querying the vector db.
document_chain.invoke(
    {
        "input":"what are the core benefits of langsmth?",
        "context": vectorstoredb.similarity_search(query)
    }
)


In [ ]:
# Method 2: Using the retriever and chaining

retriever = vectorstoredb.as_retriever() # Convert vectordb to retriever as an interface

from langchain_classic.chains import create_retrieval_chain

retrieval_chain =  create_retrieval_chain(retriever,document_chain)
response = retrieval_chain.invoke({"input":"what are the core benefits of langsmth?"})
response


Let's add chat history

In [ ]:
from langchain_classic.chains import create_history_aware_retriever # the retriever knows the history of the conversation 
from langchain_core.prompts import MessagesPlaceholder # use this to store messages or chat history


contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("user","{input}")
    ]
)

In [ ]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_prompt) # retriever will consider the chat history
history_aware_retriever

In [ ]:
# New chains as we have updated with Chat History

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the following based on the context.<context>{context}</context>"),
        MessagesPlaceholder("chat_history"),
        ("user", "{input}"),
    ]
)

new_document_chain = create_stuff_documents_chain(llm, qa_prompt) ## This combines and passes the docs inform of context
new_retrieval_chain =  create_retrieval_chain(history_aware_retriever,new_document_chain)



In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

## Method 1: We are mnaually using a list to maintain chat history and appending after every LLM call.

chat_history = []
question = "what is Langsmith?"

response1 = new_retrieval_chain.invoke({"input": question,"chat_history":chat_history})
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

#response1
question2 = "what are it's benefits?"
response2 = new_retrieval_chain.invoke({"input": question2,"chat_history":chat_history})
response2


In [ ]:
chat_history

In [ ]:
## Method 2 : Using a more better way to use RunnableWithMessageHistory

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {} # this data stricture will store the session ids and chat message history

# This function will be used to track the history per session as there can be multiple chat sessions
def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    new_retrieval_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history", #--> This name is from MessagesPlaceholder("chat_history") defined in prompt
    output_messages_key="answer",
    )




In [ ]:
conversational_rag_chain.invoke(
    {"input":"how to enable it?"},
    config={"configurable":{"session_id":"abc1234"}}
)["answer"]


In [35]:
hist= get_session_history("abc1234")
hist.messages

[HumanMessage(content='how to enable it?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="To enable LangChain and start building your own agents and applications, you can follow these general steps:\n\n1. **Installation**: Follow the installation instructions available in the documentation. This typically involves using a package manager (like pip) to install LangChain.\n\n2. **Quickstart Guide**: After installation, refer to the Quickstart guide provided in the documentation. This guide will walk you through the initial setup process and help you create your first agent.\n\n3. **Explore Core Features**: Familiarize yourself with the core benefits of LangChain, such as its pre-built agent architecture, integration capabilities, and features like durability, persistence, and human-in-the-loop support.\n\n4. **Development**: Begin developing your agents using the APIs and tools provided by LangChain. You can also use LangSmith for debugging and gaining insights into you